In [ ]:
def meiosis_with_recombination(
    diploid_pair: 'DiploidChromosomePair', # Assuming DiploidChromosomePair is defined
    recomb_event_probabilities: dict,
    recomb_probabilities: list # Length = num_loci_per_chromosome (probabilities BETWEEN loci)
) -> 'Chromosome': # Assuming Chromosome is defined
    """
    Simulates the process of meiosis and recombination for a single diploid chromosome pair
    to produce a single haploid recombinant chromosome.

    This function determines the number of crossover events based on provided probabilities
    and then, if crossovers occur, selects their positions along the chromosome based on
    locus-specific recombination probabilities. Finally, it constructs a new recombinant
    chromosome by combining segments from the two homologous chromatids of the input pair.

    Args:
        diploid_pair (DiploidChromosomePair): The pair of homologous chromosomes (two Chromosome objects)
                                             that will undergo meiosis and recombination.
        recomb_event_probabilities (dict): A dictionary where keys are the number of
                                           recombination events (0, 1, 2, etc.) and values
                                           are their respective probabilities. For example,
                                           `{0: 0.1, 1: 0.85, 2: 0.05}` means 10% chance of 0 crossovers,
                                           85% chance of 1 crossover, and 5% chance of 2 crossovers.
        recomb_probabilities (list): A list of floating-point probabilities, where each value
                                     represents the probability of a recombination event occurring
                                     in the interval *between* consecutive loci. Its length should
                                     be `num_loci_per_chromosome - 1` to cover all intervals.

    Returns:
        Chromosome: A newly generated `Chromosome` object representing one of the
                    haploid products of meiosis, which may be recombinant.
    """
    loci_len = len(diploid_pair.chromatid1.alleles) # Get the total number of loci on the chromosome

    # 1. Decide the number of recombination events (crossovers)
    # Uses random.choices to pick 0, 1, or 2 events based on their defined probabilities.
    n_events = random.choices(
        population=[0, 1, 2], # The possible number of events
        weights=[recomb_event_probabilities.get(i, 0) for i in [0, 1, 2]], # Get probabilities for each event count
        k=1 # Pick one outcome
    )[0] # Extract the single chosen number of events from the list

    # 2. Determine possible breakpoint positions
    # Possible breakpoints occur *between* loci. For `loci_len` loci, there are `loci_len - 1`
    # intervals, corresponding to positions 1 through `loci_len - 1`.
    possible_positions = list(range(1, loci_len))
    chosen_positions = [] # This list will store the specific locus indices where crossovers occur

    # 3. Handle recombination events (if any)
    if n_events > 0:
        # Extract the relevant recombination probabilities for the intervals between loci.
        # This assumes `recomb_probabilities` is a list of length `num_loci_per_chromosome - 1`
        # or longer, and we only need the probabilities up to the last interval.
        weights = recomb_probabilities[:loci_len-1]
        weights_sum = sum(weights)

        if weights_sum == 0:
            # If all recombination probabilities are zero, choose breakpoints uniformly at random
            # This is a fallback to ensure the simulation doesn't halt if probabilities are misconfigured.
            if len(possible_positions) < n_events:
                chosen_positions = possible_positions[:] # Take all available positions if not enough for n_events
            else:
                chosen_positions = sorted(random.sample(possible_positions, n_events)) # Choose unique positions randomly
        else:
            # Weighted random sampling of breakpoint positions without replacement
            # This loop ensures that `n_events` distinct breakpoint positions are chosen
            # based on the `recomb_probabilities` (weights).
            num_events_to_pick = min(n_events, len(possible_positions)) # Don't try to pick more than available
            while len(chosen_positions) < num_events_to_pick:
                # `random.choices` is used to pick a position based on weights.
                # The selected position is then checked for uniqueness and added.
                pos = random.choices(possible_positions, weights=weights, k=1)[0]
                if pos not in chosen_positions:
                    chosen_positions.append(pos)
            chosen_positions.sort() # Sort chosen positions for easier segment construction

        # 4. Construct the recombinant chromatid
        # Start the recombinant chromosome by choosing a random chromatid from the diploid pair.
        current_chromatid_source = random.choice([diploid_pair.chromatid1.alleles, diploid_pair.chromatid2.alleles])
        # The other chromatid is the alternate source for segments after a crossover.
        other_chromatid_source = diploid_pair.chromatid1.alleles if current_chromatid_source is diploid_pair.chromatid2.alleles else diploid_pair.chromatid2.alleles

        recombinant_alleles = [] # This list will build the new chromosome's allele sequence
        last_pos = 0 # Keeps track of the start of the current segment
        # Append the end of the chromosome as a final breakpoint to ensure the last segment is copied.
        breakpoints = chosen_positions + [loci_len]

        for pos in breakpoints:
            # Copy the segment from the current source chromatid up to the current breakpoint.
            recombinant_alleles.extend(current_chromatid_source[last_pos:pos])
            # Switch the source chromatids for the next segment (after a crossover).
            current_chromatid_source, other_chromatid_source = other_chromatid_source, current_chromatid_source
            last_pos = pos # Update the start position for the next segment

        return Chromosome(recombinant_alleles) # Return the newly formed recombinant chromosome

    else: # n_events == 0 (no recombination occurs)
        # If there are no crossovers, the offspring chromatid is simply a copy of one
        # of the parental chromatids, chosen randomly with 50/50 probability.
        if random.random() < 0.5: # random.random() returns a float in [0.0, 1.0)
            return Chromosome(diploid_pair.chromatid1.alleles)
        else:
            return Chromosome(diploid_pair.chromatid2.alleles)